# 🧠 Support Vector Machines (SVM) Explanation and Hands-on Example

Welcome to the hands-on explanation notebook for **Support Vector Machines (SVM)**! In this notebook, we will:
1. Generate a linearly separable dataset and fit a **Linear SVM** to find the maximum margin separating hyperplane.
2. Visualize the support vectors, decision boundary, and margin lines.
3. Generate a concentric circular dataset to show how linear boundaries fail, and apply the **RBF Kernel Trick** to separate them.
4. Implement a **Linear SVM from scratch** using Gradient Descent on the **Soft-Margin Hinge Loss**:
   $$J(\mathbf{w}, b) = \frac{1}{2} \|\mathbf{w}\|^2 + \frac{C}{m} \sum_{i=1}^{m} \max\left(0, 1 - y^{(i)}(\mathbf{w}^T \mathbf{x}^{(i)} + b)\right)$$
5. Train our scratch SVM and visualize its boundary and margin lines.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_blobs, make_circles
from sklearn.metrics import accuracy_score

# Set seed for reproducibility
np.random.seed(42)

## 1. Linear SVM and Support Vectors

Let's generate a linearly separable 2D dataset representing two types of metal rod classes: `polished-rod` (Class +1) and `valve-stem` (Class -1).

In [ ]:
# Generate 40 separable points
X_lin, y_lin = make_blobs(n_samples=40, centers=2, random_state=6, cluster_std=0.60)
# Convert target labels to -1 and +1 for SVM formulation
y_lin = np.where(y_lin == 0, -1, 1)

# Plot the dataset
plt.figure(figsize=(8, 5))
plt.scatter(X_lin[y_lin == 1, 0], X_lin[y_lin == 1, 1], color='blue', label='Class +1: Polished Rod', s=50)
plt.scatter(X_lin[y_lin == -1, 0], X_lin[y_lin == -1, 1], color='red', label='Class -1: Valve Stem', s=50)
plt.xlabel('Visual Feature 1')
plt.ylabel('Visual Feature 2')
plt.title('Separable Metal Rod Dataset')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

Now we fit a Linear SVM classifier using `scikit-learn` with a large $C$ (Hard Margin) to visualize the boundary, margin lines, and support vectors.

In [ ]:
# Fit Linear SVM
svm_lin = SVC(kernel='linear', C=10.0)
svm_lin.fit(X_lin, y_lin)

# Retrieve weights and bias
w_sk = svm_lin.coef_[0]
b_sk = svm_lin.intercept_[0]
support_vecs = svm_lin.support_vectors_

# Function to plot decision boundary and margins
def plot_svm_boundary(model, X, y, support_vectors=None):
    plt.scatter(X[y == 1, 0], X[y == 1, 1], color='blue', s=40, label='Class +1')
    plt.scatter(X[y == -1, 0], X[y == -1, 1], color='red', s=40, label='Class -1')
    
    ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    
    xx = np.linspace(xlim[0], xlim[1], 30)
    yy = np.linspace(ylim[0], ylim[1], 30)
    YY, XX = np.meshgrid(yy, xx)
    xy = np.vstack([XX.ravel(), YY.ravel()]).T
    Z = model.decision_function(xy).reshape(XX.shape)
    
    ax.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1], alpha=0.9, linestyles=['--', '-', '--'])
    
    if support_vectors is not None:
        ax.scatter(support_vectors[:, 0], support_vectors[:, 1], s=120,
                   linewidth=1.5, facecolors='none', edgecolors='black', label='Support Vectors')

plt.figure(figsize=(8, 5))
plot_svm_boundary(svm_lin, X_lin, y_lin, support_vecs)
plt.title('Linear SVM: Max Margin Classifier')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Non-linear classification: The Kernel Trick

If our dataset is concentric (non-separable linearly), a straight line classifier performs poorly. Let's generate a circle dataset and apply the **Radial Basis Function (RBF)** kernel.

In [ ]:
# Generate concentric circles
X_circle, y_circle = make_circles(n_samples=100, factor=0.3, noise=0.08, random_state=42)
y_circle = np.where(y_circle == 0, -1, 1)

# Fit OLS/Linear SVM vs RBF SVM
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_rbf.fit(X_circle, y_circle)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Linear SVM on concentric circles
plt.sca(axes[0])
svm_linear_circle = SVC(kernel='linear', C=1.0)
svm_linear_circle.fit(X_circle, y_circle)
plot_svm_boundary(svm_linear_circle, X_circle, y_circle)
plt.title('Linear SVM Boundary (Underfitting)')

# Plot RBF Kernel SVM on concentric circles
plt.sca(axes[1])
plot_svm_boundary(svm_rbf, X_circle, y_circle)
plt.title('RBF Kernel SVM Boundary (Ideal Non-linear Separation)')

plt.show()

## 3. Support Vector Machine from Scratch (Subgradient Descent)

We will implement a soft-margin Linear SVM from scratch. We minimize the cost function:
$$J(\mathbf{w}, b) = \frac{1}{2} \|\mathbf{w}\|^2 + \frac{C}{m} \sum_{i=1}^{m} \max\left(0, 1 - y^{(i)}(\mathbf{w}^T \mathbf{x}^{(i)} + b)\right)$$

Using subgradient descent on each sample $i$:
*   If $y^{(i)} (\mathbf{w}^T \mathbf{x}^{(i)} + b) \ge 1$ (sample is correctly classified outside the margin):
    $$\mathbf{w} \leftarrow \mathbf{w} - \alpha \cdot \frac{2 \mathbf{w}}{m}$$
*   Else (sample is misclassified or violates the margin):
    $$\mathbf{w} \leftarrow \mathbf{w} - \alpha \cdot \left( \frac{2 \mathbf{w}}{m} - C \cdot y^{(i)} \mathbf{x}^{(i)} \right)$$
    $$b \leftarrow b + \alpha \cdot C \cdot y^{(i)}$$

Let's implement this!

In [ ]:
class CustomSVM:
    def __init__(self, C=1.0, learning_rate=0.001, epochs=1000):
        self.C = C
        self.lr = learning_rate
        self.epochs = epochs
        self.w = None
        self.b = 0.0

    def fit(self, X, y):
        m, n = X.shape
        self.w = np.zeros(n)
        self.b = 0.0
        
        for epoch in range(self.epochs):
            for idx, x_i in enumerate(X):
                condition = y[idx] * (np.dot(x_i, self.w) + self.b) >= 1
                if condition:
                    self.w -= self.lr * (2 * (1 / self.epochs) * self.w)
                else:
                    self.w -= self.lr * (2 * (1 / self.epochs) * self.w - self.C * y[idx] * x_i)
                    self.b += self.lr * self.C * y[idx]

    def decision_function(self, X):
        return np.dot(X, self.w) + self.b

    def predict(self, X):
        return np.sign(self.decision_function(X))

# Train our custom SVM on the linear separable dataset
custom_svm = CustomSVM(C=10.0, learning_rate=0.01, epochs=800)
custom_svm.fit(X_lin, y_lin)

# Evaluate predictions
preds_custom = custom_svm.predict(X_lin)
print(f"Custom SVM Accuracy: {accuracy_score(y_lin, preds_custom) * 100:.2f}%")
print("Weights:", custom_svm.w, "| Bias:", custom_svm.b)

## 4. Visualizing Custom SVM Boundaries

Let's visualize the boundary and margins of our Custom Scratch Linear SVM.

In [ ]:
plt.figure(figsize=(8, 5))
plot_svm_boundary(custom_svm, X_lin, y_lin)
plt.title('Custom Scratch Linear SVM Decision Boundary & Margins')
plt.grid(True, alpha=0.3)
plt.show()

## 💡 Connection to Deep Learning & YOLO
*   **Hinge Loss:** The margin loss optimized by SVMs ($\max(0, 1 - y \cdot f(x))$) is closely related to the loss functions used in multi-class classification or robust regression (such as Smooth L1 / Huber Loss used for YOLO bounding box regression).
*   **SVM on CNN Embeddings:** Historically, deep models like R-CNN used a convolutional neural network to extract features, and then trained separate SVM classifiers on top of the feature representations to identify objects. Modern networks train everything end-to-end using fully connected classification layers, but the margin classification theory remains a foundational concept.